In [4]:
#D1
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch

features = torch.arange(10 * 2).reshape(10, 2).float()
labels = torch.arange(10)
ds = TensorDataset(features, labels)
print(len(ds), ds[3])

10 (tensor([6., 7.]), tensor(3))


In [11]:
#D2: serves batch, no shuffle
loader = DataLoader(ds, batch_size=4, shuffle=False, drop_last=True)
print(len(loader))
for i, (xb, yb) in enumerate(loader):
  print(i, xb.shape, yb)

3
0 torch.Size([4, 2]) tensor([0, 1, 2, 3])
1 torch.Size([4, 2]) tensor([4, 5, 6, 7])
2 torch.Size([2, 2]) tensor([8, 9])


In [12]:
#D3: shuffle
torch.manual_seed(0)
sh = DataLoader(ds, batch_size=4, shuffle=True)
for epoch in range(2):
  print("epoch", epoch, [yb.tolist() for _, yb in sh])

epoch 0 [[6, 7, 1, 4], [2, 0, 9, 8], [3, 5]]
epoch 1 [[2, 4, 7, 0], [8, 9, 5, 3], [6, 1]]


In [13]:
#Cell D4: your own Dataset class (this is the real PyTorch pattern)
class PairDataset(Dataset):
  def __init__(self, x, y):
    self.x = x
    self.y = y

  def __len__(self):
    return len(self.x)

  def __getitem__(self, i):
    return self.x[i], self.y[i]
pd2 = PairDataset(features, labels)
print(len(pd2), pd2[3])

10 (tensor([6., 7.]), tensor(3))


In [15]:
class Bad(Dataset):
  def __init__(self, x, y):
    self.x = x
    self.y = y

  def __getitem__(self, i):
    return self.x[i], self.y[i]

for b in DataLoader(Bad(features, labels), batch_size=4, shuffle=True):
    pass


TypeError: object of type 'Bad' has no len()

In [18]:
#Cell D6: autograd meets a batch (a small bridge from Hour 2)
xb, yb = next(iter(loader))
w = torch.zeros(2, requires_grad=True)
loss = ((xb @ w - yb) ** 2).mean()
loss.backward()
print(loss.item(), w.grad)

3.5 tensor([-14., -17.])
